# 01. 시세 → 커브 → 시장

| 단계 | 코드 |
|---|---|
| 시세 읽기 | `src/quanttool/market/quotes.py` `read_quotes` |
| 커브 레시피·관례 | `configs/market.yaml` (curves.UST), `configs/conventions.yaml`, `src/quanttool/config.py` |
| 부트스트랩 | `src/quanttool/market/curves.py` `build_curve` (bill BEY·par bond 헬퍼) |
| 시장 | `src/quanttool/market/market.py` `build_market`, `Market` |

> 데이터: `tests/fixtures/market_sample.txt`는 **합성 샘플**(`날짜 키 값`)이다. 흐름을 익히는 용도이고 숫자로 시장을 해석하지 않는다.

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd
import QuantLib as ql

from quanttool.config import load_config
from quanttool.market import build_market
from quanttool.market.quotes import read_quotes

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
cfg = load_config(ROOT / "configs")
quotes = read_quotes(ROOT / "tests" / "fixtures" / "market_sample.txt")  # 합성 샘플
ASOF = date(2026, 10, 23)

q = quotes[ASOF]
pd.Series(q).head()

## 1. QuantLib으로 직접: 헬퍼 → 부트스트랩
bill(1Y 이하)은 BEY → 할인채 가격, 쿠폰채는 par yield = 쿠폰이고 가격 100인 채권을 헬퍼로 쓴다.

In [ ]:
from quanttool.market.curves import bill_price

today = ql.Date(23, 10, 2026)
ql.Settings.instance().evaluationDate = today
cal = ql.UnitedStates(ql.UnitedStates.GovernmentBond)
settle = cal.advance(today, 1, ql.Days)
helpers = []
for t in ["1M", "2M", "3M", "4M", "6M", "1Y"]:
    mat = cal.adjust(settle + ql.Period(t), ql.Following)
    zcb = ql.ZeroCouponBond(1, cal, 100.0, mat, ql.Following, 100.0, settle)
    helpers.append(
        ql.BondHelper(
            ql.QuoteHandle(
                ql.SimpleQuote(bill_price(q[f"BOND/YIELD/USD/UST/PAR/{t}"], mat - settle))
            ),
            zcb,
        )
    )
for t in ["2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]:
    sch = ql.Schedule(
        settle,
        settle + ql.Period(t),
        ql.Period(ql.Semiannual),
        cal,
        ql.Unadjusted,
        ql.Unadjusted,
        ql.DateGeneration.Backward,
        False,
    )
    helpers.append(
        ql.FixedRateBondHelper(
            ql.QuoteHandle(ql.SimpleQuote(100.0)),
            1,
            100.0,
            sch,
            [q[f"BOND/YIELD/USD/UST/PAR/{t}"]],
            ql.ActualActual(ql.ActualActual.Bond),
            ql.Unadjusted,
            100.0,
            settle,
        )
    )
direct = ql.PiecewiseLogCubicDiscount(today, helpers, ql.Actual365Fixed())
direct.zeroRate(10.0, ql.Continuous).rate()

## 2. 같은 일을 QuantTool로: `build_market`
설정의 모든 커브를 만들어 **이름으로** 묶는다. 상품은 `market['UST']`에 연결된다.

In [ ]:
market = build_market(ASOF, q, cfg)
print("시장 객체:", list(market.objects))
with market.active():
    print(
        "직접 vs QuantTool 10Y 할인계수 차이:", market["UST"].discount(10.0) - direct.discount(10.0)
    )

## 3. 진단과 커브 표

In [ ]:
from quanttool.market.curves import curve_shape, curve_table, helper_repricing

display(helper_repricing(market.built["UST"], ASOF))
with market.active():
    table = curve_table(market["UST"], cfg.conventions["UST"])
display((table[["par", "zero_cc", "fwd_1y"]] * 100).round(3))
curve_shape(table)

## 4. 시장 객체가 하는 일: 요인(흔들 수 있는 값)
`market['UST']`는 부트스트랩한 커브 **자체가 아니다**. 그 위에 격자 만기별 제로 이동량 층을 한 겹 씌운 커브다 (`market/curves.py` `CurveFactors`).

```
market['UST'] = PiecewiseZeroSpreadedTermStructure(
    부트스트랩 커브 (market.built['UST'].curve),
    격자 만기별 제로 이동량 Quote  ← 요인. 처음엔 전부 0
)
```

- 이동량이 0이면 부트스트랩 커브와 **정확히** 같다 (기준 가격을 바꾸지 않는다).
- 시나리오 `{'UST/5Y': 0.0001}`은 5Y 이동량 Quote 하나를 1bp로 바꾼다. 격자점 사이는 선형이라 5Y에서 1bp, 4.5Y(4Y와 5Y 가운데)에서 0.5bp, 이웃 격자점 4Y·6Y에서 0이다. `with` 블록을 나가면 0으로 돌아온다.
- 채권 A와 B가 모두 `market['UST']`에 연결돼 있으면, Quote 하나를 바꿔도 QuantLib Observer가 둘 다 다시 가격 낸다. 커브를 다시 만들거나 상품을 다시 조립하지 않는다. key rate는 이것을 만기마다 반복할 뿐이다.

In [ ]:
f = market.factors()
print(
    len(f),
    "개 요인 (시장 전체: UST·KOFR·CD91·SPX …), 예:",
    [k for k in f if k.startswith("UST/")][:4],
)
base = market.built["UST"].curve


def zero_bp(h, years):
    return h.zeroRate(years, ql.Continuous).rate() * 1e4


with market.active():
    print("기준 상태 차이 (7.3Y 할인계수):", market["UST"].discount(7.3) - base.discount(7.3))
    t = market.objects["UST"].times  # 격자 만기 → 연 단위 시간
    with market.scenario({"UST/5Y": 0.0001}):
        at = {
            "4Y": t["4Y"],
            "4.5Y (4Y·5Y 가운데)": (t["4Y"] + t["5Y"]) / 2,
            "5Y": t["5Y"],
            "6Y": t["6Y"],
        }
        moved = {k: zero_bp(market["UST"], x) - zero_bp(base, x) for k, x in at.items()}
    print("블록 밖 5Y 이동량:", f["UST/5Y"].value())
pd.Series(moved, name="5Y +1bp → 제로 이동 (bp)").round(4)

## 5. 얼린 커브 vs 시세 연결
기본(`linked=False`)은 부트스트랩 직후 얼려 시세·평가일과 무관하다. `linked=True`면 시세 Quote를 바꾸면 커브가 따라온다.

In [ ]:
for linked in [False, True]:
    m = build_market(ASOF, q, cfg, linked=linked)
    s = m.built["UST"].quotes()["BOND/YIELD/USD/UST/PAR/10Y"]
    with m.active():
        before = m["UST"].zeroRate(10.0, ql.Continuous).rate()
        s.setValue(s.value() + 0.0010)
        print(
            f"linked={linked}: 10Y 시세 +10bp → 10Y 제로 {(m['UST'].zeroRate(10.0, ql.Continuous).rate() - before) * 1e4:.2f}bp"
        )